# Multiple Input and Multiple Output Channels

In [1]:
import torch
from d2l import torch as d2l

## Multiple Input Channels

In [2]:
def corr2d_multi_in(X, K):
# Iterate through the 0th dimension (channel) of K first, then add them up
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

In [20]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
X, "", K

(tensor([[[0., 1., 2.],
          [3., 4., 5.],
          [6., 7., 8.]],
 
         [[1., 2., 3.],
          [4., 5., 6.],
          [7., 8., 9.]]]),
 '',
 tensor([[[0., 1.],
          [2., 3.]],
 
         [[1., 2.],
          [3., 4.]]]))

In [4]:
corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

In [10]:
def test_xk(X, K):
    for x, k in zip(X, K):
        print(x, '\n', k, '\n')    
        
test_xk(X, K)

tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]]) 
 tensor([[0., 1.],
        [2., 3.]]) 

tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]]) 
 tensor([[1., 2.],
        [3., 4.]]) 



## Multiple Output Channels

In [11]:
def corr2d_multi_in_out(X, K):
    # Iterate through the 0th dimension of K, and each time, perform
    # cross-correlation operations with input X. All of the results are
    # stacked together
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [21]:
torch.stack((K, K + 1, K + 2), 1), torch.stack((K, K + 1, K + 2), 1).size()

(tensor([[[[0., 1.],
           [2., 3.]],
 
          [[1., 2.],
           [3., 4.]],
 
          [[2., 3.],
           [4., 5.]]],
 
 
         [[[1., 2.],
           [3., 4.]],
 
          [[2., 3.],
           [4., 5.]],
 
          [[3., 4.],
           [5., 6.]]]]),
 torch.Size([2, 3, 2, 2]))

In [22]:
torch.stack((K, K + 1, K + 2), 2), torch.stack((K, K + 1, K + 2), 2).size()

(tensor([[[[0., 1.],
           [1., 2.],
           [2., 3.]],
 
          [[2., 3.],
           [3., 4.],
           [4., 5.]]],
 
 
         [[[1., 2.],
           [2., 3.],
           [3., 4.]],
 
          [[3., 4.],
           [4., 5.],
           [5., 6.]]]]),
 torch.Size([2, 2, 3, 2]))

In [23]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape, '', K

(torch.Size([3, 2, 2, 2]),
 '',
 tensor([[[[0., 1.],
           [2., 3.]],
 
          [[1., 2.],
           [3., 4.]]],
 
 
         [[[1., 2.],
           [3., 4.]],
 
          [[2., 3.],
           [4., 5.]]],
 
 
         [[[2., 3.],
           [4., 5.]],
 
          [[3., 4.],
           [5., 6.]]]]))

In [24]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

### 1 x 1 Covolutional Layer

In [35]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

In [36]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

In [39]:
X, '', K

(tensor([[[-0.5278, -1.1826,  1.3501],
          [ 1.1655,  1.9382,  0.0536],
          [-0.7957, -0.2086, -1.6138]],
 
         [[ 0.3288,  1.0152, -0.0640],
          [ 0.8493, -0.4185,  0.1511],
          [ 2.2862,  1.2631,  0.0300]],
 
         [[ 0.3645,  0.7285,  1.7037],
          [ 2.0218, -1.2792, -0.7703],
          [ 1.2760, -0.9168,  0.3368]]]),
 '',
 tensor([[[[-1.1898]],
 
          [[-0.5284]],
 
          [[-0.2599]]],
 
 
         [[[-1.9742]],
 
          [[ 0.7770]],
 
          [[-1.1523]]]]))

In [40]:
def test_1x1(X, K):
    c_i, h, w = X.shape
    print(c_i, h, w)
    c_o = K.shape[0]
    print(c_o)
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    print(X, '\n', K)
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K, X)
    print(Y)
    return Y.reshape((c_o, h, w))

In [41]:
Y_test = test_1x1(X, K)

3 3 3
2
tensor([[-0.5278, -1.1826,  1.3501,  1.1655,  1.9382,  0.0536, -0.7957, -0.2086,
         -1.6138],
        [ 0.3288,  1.0152, -0.0640,  0.8493, -0.4185,  0.1511,  2.2862,  1.2631,
          0.0300],
        [ 0.3645,  0.7285,  1.7037,  2.0218, -1.2792, -0.7703,  1.2760, -0.9168,
          0.3368]]) 
 tensor([[-1.1898, -0.5284, -0.2599],
        [-1.9742,  0.7770, -1.1523]])
tensor([[ 0.3595,  0.6814, -2.0152, -2.3609, -1.7526,  0.0566, -0.5929, -0.1809,
          1.8167],
        [ 0.8774,  2.2839, -4.6783, -3.9709, -2.6775,  0.8993,  1.8768,  2.4496,
          2.8212]])


In [42]:
Y_test

tensor([[[ 0.3595,  0.6814, -2.0152],
         [-2.3609, -1.7526,  0.0566],
         [-0.5929, -0.1809,  1.8167]],

        [[ 0.8774,  2.2839, -4.6783],
         [-3.9709, -2.6775,  0.8993],
         [ 1.8768,  2.4496,  2.8212]]])